# device-consistent-construct — worked example 3: Allocate an identity matrix on the operand device for a matmul

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `device-consistent-construct`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

`t.eye(n)` defaults to CPU `float32`. When it is multiplied against a tensor on another device or in another dtype, PyTorch raises a device error (or upcasts). Building it as `t.eye(n, device=x.device, dtype=x.dtype)` makes the identity born consistent with the operand, so the matmul is well-typed everywhere.

## Worked solution

**Goal:** project a batch of feature vectors through an identity matrix (a no-op linear map used to audit device/dtype handling).

**Step 1 — find the feature dimension and the operand attributes.** For `x` of shape `(batch, n)`, the identity must be `(n, n)`. The matmul `x @ I` requires `I` to share `x.device` and `x.dtype`.

**Step 2 — build eye with both kwargs.** `I = t.eye(n, device=x.device, dtype=x.dtype)`. The `device=` kwarg avoids the default CPU allocation; the `dtype=` kwarg avoids the default `float32`, so a `float64` input stays `float64` through the multiply.

**Step 3 — matmul.** `x @ I` returns `x` mathematically. If `I` had been default CPU `float32`, then on a `float64` input PyTorch would upcast (or on GPU it would error). With the kwargs threaded, the output dtype equals `x.dtype` and the values are unchanged.

**Why it works:** the identity is constructed in the same device+dtype universe as `x`, so the contraction never needs a corrective transfer or promotion.

In [ ]:
def identity_project(x):
    n = x.shape[-1]
    I = t.eye(n, device=x.device, dtype=x.dtype)
    return x @ I

for dt in (t.float32, t.float64):
    t.manual_seed(0)
    x = t.randn(5, 4).to(dt)
    y = identity_project(x)
    print(dt, y.dtype, t.allclose(y, x))